# Installing Dependencies

In [1]:
%pip install lightgbm scikit-learn numpy pandas scipy


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = '.'

train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

# EEG excluded: never in top-20 importance, ~100 noise dims for N=1300

for df in [train_labels, test_labels,
           trainbvp, traineda, traintemp, trainhr, trainibi, trainacc,
           testbvp,  testeda,  testtemp,  testhr,  testibi,  testacc]:
    df['timestamp'] = pd.to_numeric(df['timestamp'])

for df in [trainacc, testacc]:
    df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

print('Data loaded.')
print('Train labels:', train_labels.shape, '| Test labels:', test_labels.shape)


Data loaded.
Train labels: (1456, 4) | Test labels: (1496, 4)


## 1. Session Baselines + Dead EDA Detection

In [3]:
def compute_subject_baseline(sensor_df, val_col):
    out = {}
    for pid, grp in sensor_df.groupby('pid'):
        v = grp[val_col].dropna()
        out[pid] = (v.mean(), v.std() + 1e-8)
    return out

def compute_session_bounds(label_df):
    out = {}
    for pid, grp in label_df.groupby('pid'):
        out[pid] = (grp['timestamp'].min(), grp['timestamp'].max())
    return out

def eda_dead_subjects(eda_df, threshold=0.80):
    dead = set()
    for pid, grp in eda_df.groupby('pid'):
        zr = (grp['value'] == 0).mean()
        if zr > threshold:
            dead.add(pid)
            print(f'  {pid}: EDA zero ratio={zr:.1%} → DEAD')
    return dead

bl_tr_hr   = compute_subject_baseline(trainhr,   'value')
bl_tr_eda  = compute_subject_baseline(traineda,  'value')
bl_tr_temp = compute_subject_baseline(traintemp, 'value')
bl_tr_ibi  = compute_subject_baseline(trainibi,  'value')
bl_tr_acc  = compute_subject_baseline(trainacc,  'magnitude')
bl_tr_bvp  = compute_subject_baseline(trainbvp,  'value')

bl_te_hr   = compute_subject_baseline(testhr,   'value')
bl_te_eda  = compute_subject_baseline(testeda,  'value')
bl_te_temp = compute_subject_baseline(testtemp, 'value')
bl_te_ibi  = compute_subject_baseline(testibi,  'value')
bl_te_acc  = compute_subject_baseline(testacc,  'magnitude')
bl_te_bvp  = compute_subject_baseline(testbvp,  'value')

sess_tr = compute_session_bounds(train_labels)
sess_te = compute_session_bounds(test_labels)

print('Train EDA dead sensors:')
TRAIN_EDA_DEAD = eda_dead_subjects(traineda)
print('Test EDA dead sensors:')
TEST_EDA_DEAD  = eda_dead_subjects(testeda)
print(f'Train dead: {TRAIN_EDA_DEAD} | Test dead: {TEST_EDA_DEAD}')


Train EDA dead sensors:
  70N8: EDA zero ratio=99.6% → DEAD
  Y21H: EDA zero ratio=99.4% → DEAD
Test EDA dead sensors:
Train dead: {'Y21H', '70N8'} | Test dead: set()


## 2. Feature Extraction (same as v29 — no changes)

In [4]:
from scipy.stats import skew as _sp_skew, kurtosis as _sp_kurt

WINDOWS_MS  = [2500, 5000, 10000]
ROLL_WIN_MS = 30000


def win_stats(vals, prefix):
    feat = {}
    n = len(vals)
    if n >= 2:
        feat[f'{prefix}_mean']   = np.mean(vals)
        feat[f'{prefix}_std']    = np.std(vals)
        feat[f'{prefix}_min']    = np.min(vals)
        feat[f'{prefix}_max']    = np.max(vals)
        feat[f'{prefix}_median'] = np.median(vals)
        feat[f'{prefix}_range']  = np.max(vals) - np.min(vals)
        feat[f'{prefix}_slope']  = np.polyfit(np.arange(n), vals, 1)[0]
        feat[f'{prefix}_p25']    = np.percentile(vals, 25)
        feat[f'{prefix}_p75']    = np.percentile(vals, 75)
    else:
        for s in ['mean','std','min','max','median','range','slope','p25','p75']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat


def ibi_extended(ibi_v, prefix, bl_m, bl_s):
    feat = {}
    n = len(ibi_v)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(ibi_v)
        feat[f'{prefix}_std']   = np.std(ibi_v)
        feat[f'{prefix}_rmssd'] = np.sqrt(np.mean(np.diff(ibi_v)**2))
        feat[f'{prefix}_dev']   = (np.mean(ibi_v) - bl_m) / bl_s
        feat[f'{prefix}_q25']   = np.percentile(ibi_v, 25)
        feat[f'{prefix}_q75']   = np.percentile(ibi_v, 75)
        feat[f'{prefix}_iqr']   = feat[f'{prefix}_q75'] - feat[f'{prefix}_q25']
        feat[f'{prefix}_range'] = np.max(ibi_v) - np.min(ibi_v)
        diffs = np.abs(np.diff(ibi_v))
        feat[f'{prefix}_pnn50'] = np.mean(diffs > 50) if len(diffs) > 0 else np.nan
    else:
        for s in ['mean','std','rmssd','dev','q25','q75','iqr','range','pnn50']:
            feat[f'{prefix}_{s}'] = np.nan
    if n >= 3:
        feat[f'{prefix}_skew'] = float(_sp_skew(ibi_v))
        feat[f'{prefix}_kurt'] = float(_sp_kurt(ibi_v))
    else:
        feat[f'{prefix}_skew'] = np.nan
        feat[f'{prefix}_kurt'] = np.nan
    return feat


def acc_extended(acc_v, prefix, bl_m, bl_s):
    feat = {}
    n = len(acc_v)
    if n >= 5:
        feat[f'{prefix}_mean']   = np.mean(acc_v)
        feat[f'{prefix}_std']    = np.std(acc_v)
        feat[f'{prefix}_energy'] = np.mean(acc_v**2)
        feat[f'{prefix}_dev']    = (np.mean(acc_v) - bl_m) / bl_s
        feat[f'{prefix}_max']    = np.max(acc_v)
        feat[f'{prefix}_q25']    = np.percentile(acc_v, 25)
        feat[f'{prefix}_q75']    = np.percentile(acc_v, 75)
        feat[f'{prefix}_iqr']    = feat[f'{prefix}_q75'] - feat[f'{prefix}_q25']
    else:
        for s in ['mean','std','energy','dev','max','q25','q75','iqr']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat


def nan_eda_features(feat, wl):
    for key in list(feat.keys()):
        if f'eda_{wl}' in key and key not in [f'eda_{wl}_valid', f'eda_{wl}_zero_ratio']:
            feat[key] = np.nan
    return feat


def extract_all_features(label_df, is_train,
                         hr_df, eda_df, temp_df, ibi_df, acc_df, bvp_df,
                         bl_hr, bl_eda, bl_temp, bl_ibi, bl_acc, bl_bvp,
                         sess_bounds, eda_dead_set):

    for df in [hr_df, eda_df, temp_df, ibi_df, acc_df, bvp_df]:
        df.sort_values(['pid','timestamp'], inplace=True)

    records = []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        feat = {'pid': pid, 'timestamp': ts}
        if is_train:
            feat['arousal'] = row['arousal']

        t_min, t_max = sess_bounds.get(pid, (ts, ts))
        feat['session_pos']  = (ts - t_min) / (t_max - t_min + 1e-8)
        feat['pid_eda_dead'] = 1 if pid in eda_dead_set else 0

        feat['bl_hr']   = bl_hr.get(pid,   (np.nan, 1))[0]
        feat['bl_eda']  = bl_eda.get(pid,  (np.nan, 1))[0]
        feat['bl_temp'] = bl_temp.get(pid, (np.nan, 1))[0]
        feat['bl_ibi']  = bl_ibi.get(pid,  (np.nan, 1))[0]
        feat['bl_bvp']  = bl_bvp.get(pid,  (np.nan, 1))[0]

        hr_s_pid  = hr_df[hr_df.pid == pid]
        hr_before = hr_s_pid[hr_s_pid.timestamp <= ts]['value'].values
        feat['hr_delta'] = (hr_before[-1] - hr_before[-2]
                            if len(hr_before) >= 2 else np.nan)

        def get_win(df_, col, hw):
            s = df_[df_.pid == pid]
            return s[(s.timestamp >= ts - hw) & (s.timestamp < ts + hw)][col].values

        for hw in WINDOWS_MS:
            wl = f'w{hw//1000}s'

            hr_v = get_win(hr_df, 'value', hw)
            feat.update(win_stats(hr_v, f'hr_{wl}'))
            bl_m, bl_s = bl_hr.get(pid, (np.nan, 1))
            feat[f'hr_{wl}_dev']   = (np.mean(hr_v) - bl_m) / bl_s if len(hr_v) >= 1 else np.nan
            feat[f'hr_{wl}_count'] = len(hr_v)

            eda_v = get_win(eda_df, 'value', hw)
            feat.update(win_stats(eda_v, f'eda_{wl}'))
            feat[f'eda_{wl}_count'] = len(eda_v)
            bl_m, bl_s = bl_eda.get(pid, (np.nan, 1))
            if len(eda_v) >= 1:
                zr = np.mean(eda_v == 0)
                feat[f'eda_{wl}_zero_ratio'] = zr
                feat[f'eda_{wl}_valid']       = 0 if zr > 0.5 else 1
                feat[f'eda_{wl}_dev']         = (np.mean(eda_v) - bl_m) / bl_s
                nz = eda_v[eda_v != 0]
                feat[f'eda_{wl}_nz_mean'] = np.mean(nz) if len(nz) >= 1 else np.nan
                feat[f'eda_{wl}_nz_frac'] = len(nz) / len(eda_v)
                if zr > 0.5 or pid in eda_dead_set:
                    feat = nan_eda_features(feat, wl)
            else:
                for k in ['zero_ratio','valid','dev','nz_mean','nz_frac']:
                    feat[f'eda_{wl}_{k}'] = np.nan
                feat = nan_eda_features(feat, wl)

            temp_v = get_win(temp_df, 'value', hw)
            feat.update(win_stats(temp_v, f'temp_{wl}'))
            bl_m, bl_s = bl_temp.get(pid, (np.nan, 1))
            feat[f'temp_{wl}_dev']   = (np.mean(temp_v) - bl_m) / bl_s if len(temp_v) >= 1 else np.nan
            feat[f'temp_{wl}_count'] = len(temp_v)

            ibi_v = get_win(ibi_df, 'value', hw)
            bl_m, bl_s = bl_ibi.get(pid, (np.nan, 1))
            feat.update(ibi_extended(ibi_v, f'ibi_{wl}', bl_m, bl_s))
            feat[f'ibi_{wl}_count'] = len(ibi_v)

            acc_v = get_win(acc_df, 'magnitude', hw)
            bl_m, bl_s = bl_acc.get(pid, (np.nan, 1))
            feat.update(acc_extended(acc_v, f'acc_{wl}', bl_m, bl_s))
            feat[f'acc_{wl}_count'] = len(acc_v)

            bvp_v = get_win(bvp_df, 'value', hw)
            bl_m, bl_s = bl_bvp.get(pid, (np.nan, 1))
            if len(bvp_v) >= 10:
                feat[f'bvp_{wl}_std']   = np.std(bvp_v)
                feat[f'bvp_{wl}_range'] = np.max(bvp_v) - np.min(bvp_v)
                feat[f'bvp_{wl}_iqr']   = np.percentile(bvp_v,75) - np.percentile(bvp_v,25)
                feat[f'bvp_{wl}_dev']   = (np.mean(bvp_v) - bl_m) / bl_s
            else:
                for s in ['std','range','iqr','dev']:
                    feat[f'bvp_{wl}_{s}'] = np.nan
            feat[f'bvp_{wl}_count'] = len(bvp_v)

        for sensor, df_, col in [('hr', hr_df, 'value'), ('temp', temp_df, 'value')]:
            s    = df_[df_.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)][col].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)][col].values
            feat[f'{sensor}_roll_dev'] = ((np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
                                          if len(past) >= 2 and len(cur) >= 1 else np.nan)

        if pid not in eda_dead_set:
            s    = eda_df[eda_df.pid == pid]
            past = s[(s.timestamp >= ts - ROLL_WIN_MS) & (s.timestamp < ts)]['value'].values
            cur  = s[(s.timestamp >= ts - 2500)        & (s.timestamp < ts + 2500)]['value'].values
            feat['eda_roll_dev'] = ((np.mean(cur) - np.mean(past)) / (np.std(past) + 1e-8)
                                    if len(past) >= 2 and len(cur) >= 1 else np.nan)
        else:
            feat['eda_roll_dev'] = np.nan

        feat['hr_temp_product']    = feat.get('hr_w5s_mean', np.nan) * feat.get('temp_w5s_mean', np.nan)
        feat['dev_hr_eda_product'] = feat.get('hr_w5s_dev',  np.nan) * feat.get('eda_w5s_dev',  np.nan)

        records.append(feat)

    return pd.DataFrame(records)


print('Feature extraction ready.')


Feature extraction ready.


In [5]:
print('Extracting TRAIN features...')
train_feats = extract_all_features(
    train_labels, True,
    trainhr, traineda, traintemp, trainibi, trainacc, trainbvp,
    bl_tr_hr, bl_tr_eda, bl_tr_temp, bl_tr_ibi, bl_tr_acc, bl_tr_bvp,
    sess_tr, TRAIN_EDA_DEAD
)
train_feats.insert(0, 'id', train_labels['id'].values)
print('Train features:', train_feats.shape)


Extracting TRAIN features...
Train features: (1456, 206)


In [6]:
print('Extracting TEST features...')
test_feats = extract_all_features(
    test_labels, False,
    testhr, testeda, testtemp, testibi, testacc, testbvp,
    bl_te_hr, bl_te_eda, bl_te_temp, bl_te_ibi, bl_te_acc, bl_te_bvp,
    sess_te, TEST_EDA_DEAD
)
test_feats.insert(0, 'id', test_labels['id'].values)
print('Test features:', test_feats.shape)


Extracting TEST features...
Test features: (1496, 205)


## 3. Lag Features

In [7]:
LAG_BASE = [
    'hr_w5s_mean',   'hr_w5s_dev',    'hr_roll_dev',
    'hr_delta',
    'eda_w5s_mean',  'eda_w5s_dev',   'eda_roll_dev',
    'eda_w5s_min',   'eda_w5s_p75',
    'temp_w5s_mean', 'temp_w5s_dev',  'temp_roll_dev',
    'temp_w5s_min',
    'bvp_w5s_std',
    'ibi_w5s_mean',  'ibi_w5s_std',
    'acc_w5s_max',
]
LAG_COLS = [c for c in LAG_BASE if c in train_feats.columns]


def add_lag_features(df, cols, lags=(1, 2)):
    df = df.sort_values(['pid','timestamp']).copy()
    for lag in lags:
        for col in cols:
            df[f'{col}_lag{lag}'] = df.groupby('pid')[col].shift(lag)
    for col in cols:
        df[f'{col}_roll3'] = df.groupby('pid')[col].transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
    return df


train_feats = add_lag_features(train_feats, LAG_COLS)
test_feats  = add_lag_features(test_feats,  LAG_COLS)

META_COLS = ['id','pid','timestamp','arousal']
FEAT_COLS = [c for c in train_feats.columns if c not in META_COLS]
print(f'Total features: {len(FEAT_COLS)}')


Total features: 253


## 4. Training Setup

In [8]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, classification_report, recall_score
from sklearn.preprocessing import StandardScaler

train_feats_sorted = train_feats.sort_values(['pid','timestamp']).reset_index(drop=True)
X_all  = train_feats_sorted[FEAT_COLS].values.astype(np.float32)
y_all  = (train_feats_sorted['arousal'].values - 1).astype(int)
pids   = train_feats_sorted['pid'].values
X_test = test_feats[FEAT_COLS].values.astype(np.float32)

# v30 FIX 1: Aggressive class weights for A1 and A5.
# Balanced weights alone gave A1 recall=2%, A5 recall=1% in v29.
# Only 5-7 A1/A5 samples per LOSO fold validation — model ignores them.
# Solution: multiply A1 and A5 balanced weights by 3 to force LGB to learn
# the rare-class pattern in training data even at cost of other classes.
cw_balanced = compute_class_weight('balanced', classes=np.arange(5), y=y_all)
RARE_BOOST  = 3.0
cw = cw_balanced.copy()
cw[0] *= RARE_BOOST   # A1
cw[4] *= RARE_BOOST   # A5

TRAIN_PRIOR = np.bincount(y_all, minlength=5) / len(y_all)
print('Balanced weights:', {f'A{i+1}': round(cw_balanced[i],2) for i in range(5)})
print(f'v30 weights (A1/A5 ×{RARE_BOOST}):', {f'A{i+1}': round(cw[i],2) for i in range(5)})
print('Training prior:', {f'A{i+1}': round(p,3) for i,p in enumerate(TRAIN_PRIOR)})

# v30 FIX 2: identify which features are baseline (subject-identity) columns.
# These should NOT be scaled — they encode absolute physiology per person
# and help the model know "this subject's resting HR is 85" as identity info.
BASELINE_COLS = [c for c in FEAT_COLS if c.startswith('bl_') or c == 'pid_eda_dead']
SCALE_COLS    = [c for c in FEAT_COLS if c not in BASELINE_COLS]
BL_IDX        = [FEAT_COLS.index(c) for c in BASELINE_COLS]
SC_IDX        = [FEAT_COLS.index(c) for c in SCALE_COLS]

print(f'\nFeatures to scale: {len(SCALE_COLS)} | Baseline (unscaled): {len(BASELINE_COLS)}')


Balanced weights: {'A1': np.float64(5.29), 'A2': np.float64(0.68), 'A3': np.float64(0.53), 'A4': np.float64(0.84), 'A5': np.float64(4.04)}
v30 weights (A1/A5 ×3.0): {'A1': np.float64(15.88), 'A2': np.float64(0.68), 'A3': np.float64(0.53), 'A4': np.float64(0.84), 'A5': np.float64(12.13)}
Training prior: {'A1': np.float64(0.038), 'A2': np.float64(0.295), 'A3': np.float64(0.38), 'A4': np.float64(0.237), 'A5': np.float64(0.049)}

Features to scale: 247 | Baseline (unscaled): 6


## 5. LOSO CV (v30 changes)

**Three evidence-backed fixes from v29 diagnostic:**

**Fix 1 — Aggressive rare class weights (A1/A5 × 3)**
v29 had A1 recall=2%, A5 recall=1% even with `class_weight='balanced'`.
Per LOSO fold there are only 5–7 A1 and A5 validation samples — the model
ignores them because they contribute trivially to the loss. Tripling the weight
forces LGB to spend more splits learning the A1/A5 pattern.

**Fix 2 — Exclude baseline features from StandardScaler**
v29 StandardScaler helped LIUY (+0.11) and SE4Q (+0.22) but catastrophically
hurt 70N8 (−0.15) and CQ2G (−0.08). The subjects it hurt most are those with
unusual absolute physiology (outlier HR/EDA/TEMP values). The fix: scale
derived window features (mean, std, min, max, skew…) but leave the
subject-baseline columns (bl_hr, bl_eda, bl_temp, bl_ibi, bl_bvp) in absolute
units — they serve as subject-identity signals that the model needs unmodified.

**Fix 3 — Recall-based post-hoc correction**
v29 correction was applied based on probability ratio < 1 / > 1, which
accidentally ignored A1/A5 (they had ratio < 1 from probability spreading but
still had near-zero argmax recall). New logic: boost any class whose OOF recall
< 0.20 (below random chance), by the ratio of target recall to actual recall.


In [9]:
import lightgbm as lgb

TRAIN_PIDS = sorted(train_feats_sorted['pid'].unique())
SEEDS      = [42, 7, 123, 13, 99]

oof_lgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
test_lgb = np.zeros((len(test_feats), 5), dtype=np.float64)
loso_ba  = []


def get_lgb_params(seed):
    return dict(
        objective='multiclass', num_class=5, metric='multi_logloss',
        num_leaves=31, learning_rate=0.03,
        feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
        min_child_samples=30, lambda_l1=0.5, lambda_l2=0.5,
        max_depth=6, verbose=-1, seed=seed, n_jobs=-1
    )


def scale_fold(X_tr_raw, X_va_raw, X_te_raw, sc_idx, bl_idx):
    """
    Scale only the non-baseline columns.
    Fit scaler on training fold only.
    NaN-safe: fill NaNs with column mean for fitting, restore afterwards.
    """
    scaler = StandardScaler()

    X_tr_sc = X_tr_raw[:, sc_idx].copy()
    X_va_sc = X_va_raw[:, sc_idx].copy()
    X_te_sc = X_te_raw[:, sc_idx].copy()

    col_means = np.nanmean(X_tr_sc, axis=0)
    for arr in [X_tr_sc, X_va_sc, X_te_sc]:
        nan_m = np.isnan(arr)
        arr[nan_m] = np.take(col_means, np.where(nan_m)[1])

    scaler.fit(X_tr_sc)
    X_tr_sc = scaler.transform(X_tr_sc)
    X_va_sc = scaler.transform(X_va_sc)
    X_te_sc = scaler.transform(X_te_sc)

    # Rebuild full matrices
    def rebuild(orig_raw, scaled_sc):
        out = orig_raw.copy()
        out[:, sc_idx] = scaled_sc
        return out

    X_tr = rebuild(X_tr_raw, X_tr_sc)
    X_va = rebuild(X_va_raw, X_va_sc)
    X_te = rebuild(X_te_raw, X_te_sc)
    return X_tr.astype(np.float32), X_va.astype(np.float32), X_te.astype(np.float32)


for fold_pid in TRAIN_PIDS:
    tr_mask = pids != fold_pid
    va_mask = pids == fold_pid
    X_tr_raw, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_va_raw, y_va = X_all[va_mask], y_all[va_mask]

    # Scale derived features only; baseline columns stay in absolute units
    X_tr, X_va, X_te = scale_fold(X_tr_raw, X_va_raw, X_test, SC_IDX, BL_IDX)

    sw_tr = cw[y_tr]   # aggressive A1/A5 weights
    f_lgb = np.zeros((va_mask.sum(), 5))
    t_lgb = np.zeros((len(test_feats), 5))

    for seed in SEEDS:
        dtr = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
        dva = lgb.Dataset(X_va, label=y_va, reference=dtr)
        m = lgb.train(
            get_lgb_params(seed), dtr, num_boost_round=2000,
            valid_sets=[dva],
            callbacks=[lgb.early_stopping(150, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        f_lgb += m.predict(X_va) / len(SEEDS)
        t_lgb += m.predict(X_te) / len(SEEDS)

    oof_lgb[va_mask] = f_lgb
    test_lgb        += t_lgb / len(TRAIN_PIDS)

    ba = balanced_accuracy_score(y_va, f_lgb.argmax(axis=1))
    loso_ba.append(ba)
    print(f'  {fold_pid} — BA: {ba:.4f}')

print(f'\nLOSO LGB mean: {np.mean(loso_ba):.4f} ± {np.std(loso_ba):.4f}')


  01Z2 — BA: 0.2222
  70N8 — BA: 0.2241
  7PF3 — BA: 0.2700
  CQ2G — BA: 0.1890
  D1XP — BA: 0.2295
  DT5C — BA: 0.2734
  F1ZM — BA: 0.1464
  LIUY — BA: 0.5774
  SE4Q — BA: 0.4539
  TPQI — BA: 0.2477
  Y21H — BA: 0.1014

LOSO LGB mean: 0.2668 ± 0.1297


## 6. Recall-Based Post-hoc Correction

In [10]:
oof_pred_raw = oof_lgb.argmax(axis=1)
ba_raw = balanced_accuracy_score(y_all, oof_pred_raw)
recalls = recall_score(y_all, oof_pred_raw, average=None, labels=np.arange(5),
                       zero_division=0)
print(f'OOF BA (raw argmax): {ba_raw:.4f}')
print(f'Per-class OOF recall: { {f"A{i+1}": round(r,3) for i,r in enumerate(recalls)} }')

# Boost classes whose recall is below the target (equal recall = 0.20 for 5 classes).
# Scale factor = target_recall / actual_recall, floored at 1 (never shrink a column).
# Applied to the probability column, this nudges argmax toward underperforming classes.
TARGET_RECALL = 0.20
correction = np.ones(5)
for c in range(5):
    if recalls[c] < TARGET_RECALL and recalls[c] > 0:
        correction[c] = min(TARGET_RECALL / recalls[c], 4.0)   # cap at 4x to avoid collapse
    elif recalls[c] == 0:
        # Class never predicted — use prior ratio as a conservative boost
        model_mean = oof_lgb[:, c].mean()
        correction[c] = min(TRAIN_PRIOR[c] / (model_mean + 1e-8), 4.0)

print(f'Correction factors: { {f"A{i+1}": round(correction[i],3) for i in range(5)} }')

oof_corrected  = oof_lgb  * correction
test_corrected = test_lgb * correction

ba_corrected = balanced_accuracy_score(y_all, oof_corrected.argmax(axis=1))
print(f'OOF BA (corrected):  {ba_corrected:.4f}')

recalls_corr = recall_score(y_all, oof_corrected.argmax(axis=1), average=None,
                            labels=np.arange(5), zero_division=0)
print(f'Per-class recall after: { {f"A{i+1}": round(r,3) for i,r in enumerate(recalls_corr)} }')

test_pred = test_corrected.argmax(axis=1) + 1

print('\nTest distribution:')
print(pd.Series(test_pred).value_counts().sort_index())
print('Expected from prior:', {f'A{i+1}': int(p*len(test_pred)) for i,p in enumerate(TRAIN_PRIOR)})


OOF BA (raw argmax): 0.2716
Per-class OOF recall: {'A1': np.float64(0.436), 'A2': np.float64(0.326), 'A3': np.float64(0.273), 'A4': np.float64(0.296), 'A5': np.float64(0.028)}
Correction factors: {'A1': np.float64(1.0), 'A2': np.float64(1.0), 'A3': np.float64(1.0), 'A4': np.float64(1.0), 'A5': np.float64(4.0)}
OOF BA (corrected):  0.3320
Per-class recall after: {'A1': np.float64(0.418), 'A2': np.float64(0.24), 'A3': np.float64(0.22), 'A4': np.float64(0.101), 'A5': np.float64(0.681)}

Test distribution:
1    108
2    208
3    396
4     83
5    701
Name: count, dtype: int64
Expected from prior: {'A1': 56, 'A2': 441, 'A3': 569, 'A4': 354, 'A5': 73}


## 7. OOF Diagnosis

In [11]:
oof_pred = oof_corrected.argmax(axis=1)

print('=== OOF Classification Report ===')
print(classification_report(y_all, oof_pred,
                            target_names=[f'Arousal {i+1}' for i in range(5)]))

print('Per-subject LOSO BA:')
for pid, ba in zip(TRAIN_PIDS, loso_ba):
    flag = ' ← LOW' if ba < 0.25 else ''
    print(f'  {pid}: {ba:.4f}{flag}')

print(f'\nFinal OOF BA (argmax)    : {ba_raw:.4f}')
print(f'Final OOF BA (corrected) : {ba_corrected:.4f}')
print(f'LOSO mean                : {np.mean(loso_ba):.4f} ± {np.std(loso_ba):.4f}')


=== OOF Classification Report ===
              precision    recall  f1-score   support

   Arousal 1       0.29      0.42      0.34        55
   Arousal 2       0.38      0.24      0.29       430
   Arousal 3       0.36      0.22      0.27       554
   Arousal 4       0.43      0.10      0.16       345
   Arousal 5       0.07      0.68      0.13        72

    accuracy                           0.23      1456
   macro avg       0.31      0.33      0.24      1456
weighted avg       0.36      0.23      0.25      1456

Per-subject LOSO BA:
  01Z2: 0.2222 ← LOW
  70N8: 0.2241 ← LOW
  7PF3: 0.2700
  CQ2G: 0.1890 ← LOW
  D1XP: 0.2295 ← LOW
  DT5C: 0.2734
  F1ZM: 0.1464 ← LOW
  LIUY: 0.5774
  SE4Q: 0.4539
  TPQI: 0.2477 ← LOW
  Y21H: 0.1014 ← LOW

Final OOF BA (argmax)    : 0.2716
Final OOF BA (corrected) : 0.3320
LOSO mean                : 0.2668 ± 0.1297


# Generating Final Submissions

In [12]:
submission = pd.DataFrame({
    'id':      test_feats['id'].values,
    'arousal': test_pred,
})
submission.to_csv('submission-v30.csv', index=False)
print('submission-v30.csv saved.')
print(f'Shape: {submission.shape}')
print(submission['arousal'].value_counts().sort_index())


submission-v30.csv saved.
Shape: (1496, 2)
arousal
1    108
2    208
3    396
4     83
5    701
Name: count, dtype: int64
